# Tự kiểm thử M1 trên Google Colab (CPU)

Notebook chạy Engine `src/crawl/` trên một mẫu nhỏ của Range, trong môi trường giống
testbed của đề (Colab CPU-only):

1. Clone repo đúng nhánh, ghi commit đang chạy.
2. Cài `requirements.txt`, chạy toàn bộ `pytest`.
3. Crawl **cold** `N_IDS` ID đầu Range qua harness `src.benchmark` (RAM cả cây tiến trình, disk).
4. Chạy lại cùng lệnh để kiểm **resume**: Version đã xác minh checksum không được tải lại.
5. Thống kê bằng `src.make_stats` và đối chiếu bằng `src.crawl.validation`.

**Cấu hình bí mật:** mở biểu tượng 🔑 *Secrets* ở thanh bên trái Colab, thêm
`CONTACT_EMAIL` (bắt buộc, arXiv yêu cầu User-Agent có email) và `S2_API_KEY` (tùy chọn),
bật *Notebook access*. Không dán token hay key vào ô code.

Dữ liệu crawl nằm ở `/content/...` và mất khi phiên Colab kết thúc. Muốn giữ lại thì đặt
`USE_DRIVE = True`. Không commit dữ liệu vào repo.

In [ ]:
# ==== Tham số ====
REPO_URL = "https://github.com/hcmus-arxiv-reference/arxiv-reference-matching.git"
BRANCH = "chore/colab-selftest"   # đổi thành "main" sau khi PR đã merge
MEMBER_NAME = "bao"
RANGE_PREFIX, RANGE_START, RANGE_END = "2508", 2872, 7871   # Range của Bảo (gồm hai đầu)
N_IDS = 20                        # số ID đầu Range để thử; tăng dần khi đã ổn
RUN_PYTEST = True
USE_DRIVE = False                 # True: lưu data/stats vào Google Drive để giữ qua phiên

REPO_DIR = "/content/arxiv-reference-matching"
WORK_ROOT = "/content/drive/MyDrive/arxiv-m1-selftest" if USE_DRIVE else "/content/m1-selftest"
DATA_DIR = f"{WORK_ROOT}/data"
STATS_DIR = f"{WORK_ROOT}/stats"

In [ ]:
import os, subprocess
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("branch:", BRANCH, "commit:", COMMIT)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Đọc bí mật từ Colab Secrets vào biến môi trường của tiến trình con; không in giá trị.
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

for name in ("CONTACT_EMAIL", "S2_API_KEY"):
    value = secret(name)
    if value:
        os.environ[name] = value
    else:
        os.environ.pop(name, None)
    print(f"{name}: {'đã đặt' if value else 'chưa đặt'}")
assert os.environ.get("CONTACT_EMAIL"), "Thêm CONTACT_EMAIL trong Colab Secrets trước khi crawl"

os.environ.update({"MEMBER_NAME": MEMBER_NAME, "DATA_DIR": DATA_DIR,
                   "ARXIV_RANGE_PREFIX": RANGE_PREFIX,
                   "ARXIV_RANGE_START": str(RANGE_START), "ARXIV_RANGE_END": str(RANGE_END)})

In [ ]:
# Thông tin máy để ghi vào báo cáo/benchmark.
import platform, psutil, shutil
MACHINE = (f"Colab {platform.platform()}, {os.cpu_count()} vCPU, "
           f"{psutil.virtual_memory().total / 2**30:.1f} GiB RAM, "
           f"disk trống {shutil.disk_usage('/content').free / 2**30:.0f} GiB")
print(MACHINE)
print("Python", platform.python_version())

In [ ]:
if RUN_PYTEST:
    !PYTEST_DISABLE_PLUGIN_AUTOLOAD=1 python -m pytest -q

In [ ]:
# Sinh danh sách N_IDS ID đầu Range.
os.makedirs(WORK_ROOT, exist_ok=True)
IDS_FILE = f"{WORK_ROOT}/ids.txt"
first = f"{RANGE_PREFIX}.{RANGE_START:05d}"
last = f"{RANGE_PREFIX}.{RANGE_START + N_IDS - 1:05d}"
assert RANGE_START + N_IDS - 1 <= RANGE_END
!python -m src.make_ids {first} {last} -o {IDS_FILE}
!wc -l {IDS_FILE}

## Lượt 1: crawl cold

Nếu `DATA_DIR` đã có dữ liệu từ lần trước thì lượt này là resume, không còn là cold.
Muốn đo lại từ đầu thì đặt `CLEAN = True`.
`--retry-delay 60` để lượt thử không phải chờ 30 phút giữa các pass.

In [ ]:
CLEAN = False
if CLEAN and os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
COLD = not os.path.exists(f"{DATA_DIR}/manifests/crawl.jsonl")
print("cold run:", COLD)

CRAWL = (f"python -m src.milestone1_crawler --ids-file {IDS_FILE} --data-dir {DATA_DIR} "
         f"--max-passes 2 --retry-delay 60")
!python -m src.benchmark --data-dir {DATA_DIR} --out {STATS_DIR}/run_cold -- {CRAWL}

## Lượt 2: kiểm resume

Chạy lại đúng lệnh. Kỳ vọng khi lượt 1 đã xong hết: tổng `attempts` trong Manifest
không tăng (không task nào được thử lại), `visited_this_pass` bằng 0 và lượt chạy chỉ mất vài giây.

In [ ]:
import json
from pathlib import Path
MANIFEST = Path(DATA_DIR, "manifests", "crawl.jsonl")

def attempts():
    """Tổng attempts theo bản ghi cuối của mỗi task (arxiv_id, version, kind)."""
    latest = {}
    for line in MANIFEST.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        latest[record["arxiv_id"], record["version"], record["kind"]] = record["attempts"]
    return sum(latest.values())

before = attempts()
!python -m src.benchmark --data-dir {DATA_DIR} --out {STATS_DIR}/run_resume -- {CRAWL}
after = attempts()
print(f"Manifest attempts: {before} -> {after} (tăng {after - before})")

## Thống kê và đối chiếu

In [ ]:
import json
!python -m src.make_stats --data-dir {DATA_DIR} --ids-file {IDS_FILE} --out {STATS_DIR}/m1
summary = json.loads(Path(STATS_DIR, "m1", "summary.json").read_text())
cold = json.loads(Path(STATS_DIR, "run_cold", "benchmark.json").read_text())
resume = json.loads(Path(STATS_DIR, "run_resume", "benchmark.json").read_text())
for name, run in (("cold", cold), ("resume", resume)):
    print(name, {k: run[k] for k in ("return_code", "wall_seconds", "peak_rss_mb",
                                    "avg_rss_mb", "peak_disk_mb", "final_disk_mb")})
summary

In [ ]:
# Validator nhận benchmark theo mẫu docs/benchmark/benchmark.template.json; điền các
# trường nó cần từ lượt cold, để null những gì harness không đo được.
from src.crawl.validation import validate_crawl

claim = {
    "schema_version": 1, "member": MEMBER_NAME, "branch": BRANCH, "commit": COMMIT,
    "range": {"prefix": RANGE_PREFIX, "start": RANGE_START, "end": RANGE_END},
    "ids_attempted": N_IDS,
    "run": {"cold_run": COLD, "resumed_passes": 1, "machine": MACHINE,
            "python": platform.python_version(), "s2_api_key": bool(os.environ.get("S2_API_KEY")),
            "measured_by": "src.benchmark (psutil; RSS cả cây tiến trình, disk qua os.walk)"},
    "totals": {"wall_seconds": cold["wall_seconds"], "peak_rss_mb": cold["peak_rss_mb"],
               "avg_rss_mb": cold["avg_rss_mb"], "peak_disk_mb": cold["peak_disk_mb"],
               "output_mb": resume["final_disk_mb"]},
    "notes": "Colab self-test trên mẫu nhỏ, tạo bởi notebooks/colab_m1_selftest.ipynb",
}
claim_path = Path(STATS_DIR, "benchmark.claim.json")
claim_path.write_text(json.dumps(claim, ensure_ascii=False, indent=2))
report = validate_crawl(DATA_DIR, claim_path)
print("ok:", report["ok"])
print(json.dumps(report.get("observed"), indent=2))
for item in report["findings"]:
    print("-", item["code"], item["message"])

In [ ]:
# Publication chưa xong (nếu có) và lỗi theo loại task.
import csv
rows = list(csv.DictReader(open(Path(STATS_DIR, "m1", "papers.csv"), encoding="utf-8")))
print({s: sum(r["status"] == s for r in rows) for s in sorted({r["status"] for r in rows})})
[r for r in rows if r["status"] != "done"][:20]

In [ ]:
# Tải thư mục stats về máy (không gồm dữ liệu crawl).
shutil.make_archive(f"{WORK_ROOT}/stats-{COMMIT}", "zip", STATS_DIR)
from google.colab import files
files.download(f"{WORK_ROOT}/stats-{COMMIT}.zip")